This file constitutes the first attempt at implementing a Fourier neural operator in python. It is inspired by Zongyi Li's paper "Fourier Neural Operator for Parametric Partial Differential Equations", and the implementation is taken from the corresponding repository found here:

https://github.com/ixScience/fourier_neural_operator/blob/master/fourier_1d.py

In [11]:
#Imports


import matplotlib.pyplot as plt
import os
from timeit import default_timer
from FNO1D_def import *

torch.manual_seed(0)
np.random.seed(0)

In [30]:
#Configurations

n_train = 900
n_test = 100


#Subsampling: made redundant for now
"""
sub = 2**3      #subsampling rate (?)
h = 2**13 // sub    #total grid size divided by the subsampling rate (?)
s = h   #(?)
"""

batch_size = 20
lr = 1e-3
epochs = 100
iterations = epochs * (n_train//batch_size)

modes = 2
width = 64  #these values are taken from the repo, mess around with them and see what happens
model_path = "/scratch/mnhagen/models/burgers/with_spatial/FNO1D_burgers_1024_v1.pt"

In [31]:
#Read data

sub = 8

dataloader = MatReader("/scratch/mnhagen/datasets/burgers/with_spatial/burgers_1D_256_k64.h5")
x_data = dataloader.read_field("a").permute(2,1,0)[:, ::sub]
y_data = dataloader.read_field("u").permute(1,0)[:, ::sub]

s = x_data.shape[1]
dim = x_data.shape[2]

x_train = x_data[:n_train,:]
y_train = y_data[:n_train,:]
x_test = x_data[-n_test:,:]
y_test = y_data[-n_test:,:]


print(x_data.shape)
print(y_data.shape)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)


torch.Size([1000, 32, 2])
torch.Size([1000, 32])


In [32]:
#Model - training and evaluation

model = FNO1D(modes, width).cuda()
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = 1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = iterations)

myloss = LpLoss(size_average = False)
for ep in range(epochs):
    model.train()
    t1 = default_timer()
    train_mse = 0
    train_l2 = 0
    for x,y in train_loader:
        x,y = x.cuda(), y.cuda()

        optimizer.zero_grad()
        out = model(x)

        mse = F.mse_loss(out.view(batch_size, -1), y.view(batch_size, -1), reduction='mean')
        l2 = myloss(out.view(batch_size, -1), y.view(batch_size, -1))
        l2.backward() # use the l2 relative loss

        optimizer.step()
        scheduler.step()
        train_mse += mse.item()
        train_l2 += l2.item()

    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.cuda(), y.cuda()

            out = model(x)
            test_l2 += myloss(out.view(batch_size, -1), y.view(batch_size, -1)).item()

    train_mse /= len(train_loader)
    train_l2 /= n_train
    test_l2 /= n_test

    t2 = default_timer()
    print(ep, t2-t1, train_mse, train_l2, test_l2)


124097
0 0.63107755407691 0.022761982327534094 0.7720829221937392 0.6509723472595215
1 0.5674323220737278 0.015397639676100678 0.6484774653116862 0.6465182971954345
2 0.6695951679721475 0.015360837595330344 0.6476385084788004 0.6466793823242187
3 0.5141702643595636 0.015310804297526678 0.6466330718994141 0.6470349502563476
4 0.5659398077987134 0.015348674895034897 0.6474984126620823 0.6476185607910157
5 0.4837881322018802 0.015344032293392552 0.6472332530551487 0.6457530784606934
6 0.6661884519271553 0.015351003367039892 0.647501565085517 0.6508869361877442
7 0.7996928412467241 0.015347008013890849 0.6474325444963244 0.6454968929290772
8 0.7651897240430117 0.015337255431546105 0.6473184903462728 0.6458340930938721
9 0.8606180609203875 0.015385947521362039 0.6483866744571262 0.6478890705108643
10 0.6883776392787695 0.015350725087854598 0.6471539221869574 0.6477647686004638
11 0.5369180194102228 0.015279175154864788 0.6462833415137397 0.6449026393890381
12 0.559641232714057 0.01531183206

In [33]:
#Save model

model_dir = "/scratch/mnhagen/models/burgers/with_spatial"
model_name = "FNO1D_burgers_32_modes2_k64"
model_path = os.path.join(model_dir, model_name + ".pt")
overwrite = True

os.makedirs(model_dir, exist_ok= True)

if not overwrite:
    if os.path.exists(model_path):
        raise FileExistsError(f"Model file already exists: {model_path}")

torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to /scratch/mnhagen/models/burgers/with_spatial/FNO1D_burgers_32_modes2_k64.pt


In [5]:
#Load model for testing

model = FNO1D(modes, width).cuda()
model.load_state_dict(torch.load(model_path))
model.eval()
model_name = "FNO1D_burgers_1024_v1"

myloss = LpLoss(size_average = False)
print(f"Loaded model from {model_path}")

Loaded model from /scratch/mnhagen/models/burgers/with_spatial/FNO1D_burgers_1024_v1.pt


In [7]:
#Test model and visualize predictions on test data

plot_idx = 10
print_all_pred = False

pred = torch.zeros(y_test.shape)
index = 1
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=1, shuffle=False)
with torch.no_grad():
    for x, y in test_loader:
        test_l2 = 0
        x, y = x.cuda(), y.cuda()
        #print(x.shape)

        out = model(x).view(-1)
        pred[index] = out

        test_l2 += myloss(out.view(1, -1), y.view(1, -1)).item()
        if print_all_pred:
            print(index, test_l2)
        index = index + 1

fig, ax = plt.subplots(1,3, figsize = [16,5])
a = np.linspace(0,1,s)  #change 1 to be x_data[last entry spatial dimension]

fig.suptitle(f"{model_name} prediction")

ax[0].set_title("Initial condition")
ax[0].set_ylim(-1,1)
ax[0].plot(a, x_test[plot_idx, :, 0])

ax[1].set_title("Ground truth")
ax[1].set_ylim(-1,1)
ax[1].plot(a, y_test[plot_idx, :])

ax[2].set_title("Model prediction")
ax[2].set_ylim(-1,1)
ax[2].plot(a, pred[plot_idx])

plt.show()

IndexError: index 100 is out of bounds for dimension 0 with size 100